In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import os
import time

In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [ ]:
# Constants
RANDOMSTATE = 1082
MODEL_NAME = "OneClassSVM"


In [ ]:
EXP = protocol.one_class("uav")
X_train = EXP.X_train.values
X_test = EXP.X_test.values
y_train = EXP.y_train
y_test = EXP.y_test


In [ ]:
# Train One-Class SVM on benign only
ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.04)  # nu ≈ expected outlier ratio
start_time = time.time()
ocsvm.fit(X_train)
train_time = time.time() - start_time

In [ ]:
# Predict: +1 for inliers (benign), -1 for outliers (anomaly)
y_pred_ocsvm = ocsvm.predict(X_test)
# Convert to 0=benign, 1=anomaly
y_pred = (y_pred_ocsvm == -1).astype(int)

In [ ]:
# Results
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
from eval_metrics import evaluate, summary
from gps_ids.results import append_validated

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    family="OneClass",
    y_proba=None,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=ocsvm.get_params(),
    timing=None,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
